# 02. HMM Regime Detection

Fit data-driven Gaussian HMM volatility regimes, map raw states to semantic volatility labels, and compare saved HMM labels with Quantile labels from Notebook 01.


## Execution Mode

`EXECUTION_MODE = "auto"` resolves to local mode on Windows and HPC mode on Linux/HPC markers. Manual override is supported.


In [ ]:
EXECUTION_MODE = "auto"  # allowed: "auto", "local", "hpc"
STAGE_NAME = "02_hmm_regime_detection"
from pathlib import Path
import sys, warnings
warnings.simplefilter("default")

def find_workflow_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src" / "hpc_config.py").exists():
            return candidate
    fallback = Path(r"C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\HPC workflow\HPC_Regime_and_motif_discovery")
    if (fallback / "src" / "hpc_config.py").exists():
        return fallback
    raise RuntimeError("Could not locate workflow root.")

WORKFLOW_ROOT = find_workflow_root(); SRC_DIR = WORKFLOW_ROOT / "src"
if str(SRC_DIR) not in sys.path: sys.path.insert(0, str(SRC_DIR))
from hpc_config import build_workflow_paths, detect_execution_mode, environment_info, find_project_root, load_workflow_config, output_suffix, save_config_snapshot
from io_utils import ensure_workflow_dirs, setup_stage_logger
PROJECT_ROOT = find_project_root(WORKFLOW_ROOT)
MODE = detect_execution_mode(EXECUTION_MODE, PROJECT_ROOT)
CONFIG = load_workflow_config(MODE, WORKFLOW_ROOT)
PATHS = build_workflow_paths(WORKFLOW_ROOT, PROJECT_ROOT)
ensure_workflow_dirs(PATHS)
SUFFIX = output_suffix(CONFIG)
LOGGER = setup_stage_logger(PATHS.logs, STAGE_NAME, SUFFIX)
SNAPSHOT_PATH = save_config_snapshot(CONFIG, PATHS, STAGE_NAME, MODE)
env = environment_info(PATHS)
print("Stage:", STAGE_NAME)
print("Execution mode:", MODE)
print("Project root:", PATHS.project_root)
print("Workflow root:", PATHS.workflow_root)
print("Input feature root:", PATHS.data_root)
print("Config snapshot:", SNAPSHOT_PATH)
print({"python": env["python_version"].split()[0], "platform": env["platform"], "hostname": env["hostname"], "cpu_count": env["cpu_count"], "memory_available_gb": env["memory_available_gb"], "gpu": env["gpu"]})


## Discover Feature Files

Feature parquet files are auto-discovered under `final_dataset/features` and filtered by the active run config.


In [ ]:
from data_loading import discover_feature_files
from io_utils import save_table
inventory = discover_feature_files(PATHS.project_root, CONFIG)
if inventory.empty:
    raise FileNotFoundError(f"No feature parquet files discovered under {PATHS.data_root}")
inventory_for_save = inventory.copy(); inventory_for_save["path"] = inventory_for_save["path"].astype(str)
save_table(inventory_for_save, PATHS.tables / f"input_feature_inventory{SUFFIX}.csv", PATHS.tables / f"input_feature_inventory{SUFFIX}.parquet")
print("Discovered input files:")
print(inventory_for_save[["asset", "frequency", "market", "path"]].to_string(index=False))


## Run HMM Regimes

Missing `hmmlearn` is recorded as an explicit method failure; no substitute HMM labels are created.


In [ ]:
import pandas as pd
from data_loading import apply_mode_limits, describe_feature_frame, load_feature_file
from feature_selection import choose_volatility_column, ensure_core_features
from hmm_regime import INSTALL_HINT, run_hmm_regime_detection
from io_utils import failure_record, safe_name, save_failure_log, save_table
from plotting_utils import plot_posterior_confidence, plot_price_by_regime, plot_regime_distribution, plot_transition_heatmap, plot_volatility_by_regime
from regime_utils import compare_regime_partitions, load_regime_labels

quantile_labels = load_regime_labels(WORKFLOW_ROOT, "quantile", SUFFIX)
print(f"Loaded Quantile labels for comparison: {len(quantile_labels):,}")

all_labels, all_summaries, all_transitions, all_persistence = [], [], [], []
all_model_selection, all_feature_diagnostics = [], []
dataset_summaries, failures = [], []
figure_count = 0
max_figures = int(CONFIG.get("plotting", {}).get("max_figures_per_notebook", 40))
max_points = int(CONFIG.get("plotting", {}).get("max_points_per_figure", 5000))

for meta in inventory.to_dict("records"):
    asset, frequency = meta["asset"], meta["frequency"]
    try:
        LOGGER.info("HMM regimes: %s %s", asset, frequency)
        raw = load_feature_file(meta["path"])
        df = apply_mode_limits(raw, CONFIG)
        if df.empty:
            raise ValueError("No rows remain after execution-mode limits.")
        dataset_summaries.append(describe_feature_frame(meta, df))
        print(f"\n{asset} {frequency}: rows={len(df):,}, columns={df.shape[1]}, date_range={df['timestamp'].min()} -> {df['timestamp'].max()}")
        labels, summary, transitions, persistence, model_selection, feature_diag = run_hmm_regime_detection(df, asset, frequency, CONFIG)
        all_labels.append(labels); all_summaries.append(summary); all_transitions.append(transitions); all_persistence.append(persistence); all_model_selection.append(model_selection)
        feature_diag.insert(0, "frequency", frequency); feature_diag.insert(0, "asset", asset); all_feature_diagnostics.append(feature_diag)
        stem = f"{safe_name(asset)}_{safe_name(frequency)}_hmm_regimes{SUFFIX}"
        save_table(labels, PATHS.regimes_hmm / f"{stem}.csv", PATHS.regimes_hmm / f"{stem}.parquet")

        prepared = ensure_core_features(df, rolling_window=int(CONFIG.get("quantile", {}).get("default_rolling_window", 60)))
        vol_col = choose_volatility_column(prepared)
        for regime_method in labels["regime_method"].drop_duplicates():
            method_labels = labels[labels["regime_method"] == regime_method][["timestamp", "regime_label", "regime_confidence"]]
            plot_df = prepared.merge(method_labels, on="timestamp", how="left")
            method_summary = summary[summary["regime_method"] == regime_method]
            method_transitions = transitions[transitions["regime_method"] == regime_method]
            prefix = f"02_{safe_name(asset)}_{safe_name(frequency)}_{safe_name(regime_method)}{SUFFIX}"
            if figure_count < max_figures:
                plot_price_by_regime(plot_df, "regime_label", f"{asset} {frequency} close by {regime_method}", PATHS.figures / f"{prefix}_close_by_regime.png", max_points=max_points); figure_count += 1
            if vol_col and figure_count < max_figures:
                plot_volatility_by_regime(plot_df, vol_col, "regime_label", f"{asset} {frequency} volatility by {regime_method}", PATHS.figures / f"{prefix}_vol_by_regime.png", max_points=max_points); figure_count += 1
            if figure_count < max_figures:
                plot_posterior_confidence(plot_df, f"{asset} {frequency} HMM posterior confidence", PATHS.figures / f"{prefix}_posterior_confidence.png", max_points=max_points); figure_count += 1
            if figure_count < max_figures:
                plot_regime_distribution(method_summary, f"{asset} {frequency} HMM regime distribution", PATHS.figures / f"{prefix}_distribution.png"); figure_count += 1
            if figure_count < max_figures:
                plot_transition_heatmap(method_transitions, f"{asset} {frequency} HMM transition matrix", PATHS.figures / f"{prefix}_transition_heatmap.png"); figure_count += 1
    except Exception as exc:
        LOGGER.exception("HMM regime failure for %s %s", asset, frequency)
        failures.append(failure_record(STAGE_NAME, exc, asset, frequency, {"path": str(meta["path"]), "install_hint": INSTALL_HINT}))

hmm_labels = pd.concat(all_labels, ignore_index=True) if all_labels else pd.DataFrame(columns=["timestamp", "asset", "frequency", "regime_method", "regime_label", "raw_state", "regime_confidence", "selected_n_states", "feature_set"])
hmm_summary = pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame(columns=["asset", "frequency", "regime_method", "regime_label", "observations", "share", "mean_return", "std_return", "mean_rolling_vol", "median_rolling_vol", "min_timestamp", "max_timestamp"])
hmm_transitions = pd.concat(all_transitions, ignore_index=True) if all_transitions else pd.DataFrame(columns=["asset", "frequency", "regime_method", "from_regime", "to_regime", "count", "probability"])
hmm_persistence = pd.concat(all_persistence, ignore_index=True) if all_persistence else pd.DataFrame(columns=["asset", "frequency", "regime_method", "raw_state", "regime_label", "self_transition_probability", "expected_duration_observations"])
hmm_model_selection = pd.concat(all_model_selection, ignore_index=True) if all_model_selection else pd.DataFrame(columns=["asset", "frequency", "n_states", "status", "log_likelihood", "aic", "bic", "runtime_seconds", "error", "selected"])
hmm_feature_diagnostics = pd.concat(all_feature_diagnostics, ignore_index=True) if all_feature_diagnostics else pd.DataFrame(columns=["asset", "frequency", "feature", "missing_fraction", "unique_values", "kept", "center", "scale", "scaler"])
dataset_summary = pd.DataFrame(dataset_summaries)
comparison = compare_regime_partitions(hmm_labels, quantile_labels, "hmm", "quantile") if not hmm_labels.empty and not quantile_labels.empty else pd.DataFrame()
confusion = pd.DataFrame()
if not hmm_labels.empty and not quantile_labels.empty:
    merged = hmm_labels.merge(quantile_labels, on=["asset", "frequency", "timestamp"], how="inner", suffixes=("_hmm", "_quantile"))
    if not merged.empty:
        confusion = merged.groupby(["asset", "frequency", "regime_method_hmm", "regime_method_quantile", "regime_label_hmm", "regime_label_quantile"]).size().reset_index(name="count")

save_table(hmm_labels, PATHS.regimes_hmm / f"hmm_regime_labels{SUFFIX}.csv", PATHS.regimes_hmm / f"hmm_regime_labels{SUFFIX}.parquet")
save_table(hmm_summary, PATHS.regimes_hmm / f"hmm_regime_summary{SUFFIX}.csv", PATHS.regimes_hmm / f"hmm_regime_summary{SUFFIX}.parquet")
save_table(hmm_transitions, PATHS.regimes_hmm / f"hmm_transition_matrix{SUFFIX}.csv", PATHS.regimes_hmm / f"hmm_transition_matrix{SUFFIX}.parquet")
save_table(hmm_persistence, PATHS.regimes_hmm / f"hmm_persistence_metrics{SUFFIX}.csv", PATHS.regimes_hmm / f"hmm_persistence_metrics{SUFFIX}.parquet")
save_table(hmm_model_selection, PATHS.regimes_hmm / f"hmm_model_selection{SUFFIX}.csv", PATHS.regimes_hmm / f"hmm_model_selection{SUFFIX}.parquet")
save_table(hmm_feature_diagnostics, PATHS.regimes_hmm / f"hmm_feature_diagnostics{SUFFIX}.csv", PATHS.regimes_hmm / f"hmm_feature_diagnostics{SUFFIX}.parquet")
save_table(comparison, PATHS.regimes_hmm / f"hmm_quantile_comparison{SUFFIX}.csv", PATHS.regimes_hmm / f"hmm_quantile_comparison{SUFFIX}.parquet")
save_table(confusion, PATHS.regimes_hmm / f"hmm_quantile_confusion_table{SUFFIX}.csv", PATHS.regimes_hmm / f"hmm_quantile_confusion_table{SUFFIX}.parquet")
save_table(dataset_summary, PATHS.tables / f"02_hmm_dataset_summary{SUFFIX}.csv", PATHS.tables / f"02_hmm_dataset_summary{SUFFIX}.parquet")
failure_df = save_failure_log(failures, PATHS.logs / f"02_hmm_failures{SUFFIX}.csv")

print("\nSaved HMM labels:", len(hmm_labels))
print("Saved HMM summary rows:", len(hmm_summary))
print("Saved HMM/Quantile comparison rows:", len(comparison))
print("Failures:", len(failure_df))
if hmm_labels.empty:
    print("HMM produced no labels. This is expected only if hmmlearn is missing or all HMM fits failed. Install hint:", INSTALL_HINT)
hmm_summary.head(20)
